In [ ]:
# zipfile library import kar rahe hain taake zip file ko extract kar saken
import zipfile

# civicguard_distilbert.zip file ko read mode mein open karna
with zipfile.ZipFile('civicguard_distilbert.zip', 'r') as zip_ref:
  # zip file ke tamam contents ko civicguard_distilbert folder mein extract karna
    zip_ref.extractall('civicguard_distilbert')

In [ ]:
# pandas data handling ke liye import karna
import pandas as pd
# csv module import karna taake CSV reading options use kar saken
import csv

# test dataset load karna
df_test = pd.read_csv(
    "test.csv",
    engine="python", # python parser use karna
    quoting=csv.QUOTE_MINIMAL, # normal CSV quoting handle karna
    on_bad_lines="skip" # agar koi corrupted line ho to skip kar dena
)

# test labels wali file load karna
df_labels = pd.read_csv("test_labels.csv")

# rows ki total count print karna
print("Test rows:", len(df_test))
print("Label rows:", len(df_labels))

Test rows: 153164
Label rows: 153164


In [ ]:
# test data aur labels ko 'id' column ki base par merge karna
df = pd.merge(df_test, df_labels, on="id")

# un rows ko remove karna jahan toxic label -1 ho
df = df[df["toxic"] != -1].reset_index(drop=True)

# original labels ko extract karna evaluation ke liye
true_labels = df[
    ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']
].values

# dataframe ka shape print karna
print(df.shape)

(63978, 8)


In [ ]:
# Hugging Face transformers se tokenizer aur model import karna
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# saved tokenizer load karna
tokenizer = AutoTokenizer.from_pretrained(
    "civicguard_distilbert"
)

# trained classification model load karna
model = AutoModelForSequenceClassification.from_pretrained(
    "civicguard_distilbert"
)

# confirmation message
print("Model loaded successfully!")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Model loaded successfully!


In [ ]:
# Hugging Face Dataset class import karna
from datasets import Dataset

# dataframe ko Hugging Face dataset mein convert karna
test_dataset = Dataset.from_pandas(
    df[['comment_text']]
)

# text tokenization function define karna
def tokenize_function(examples):
    return tokenizer(
        examples["comment_text"], # comment text tokenize karna
        padding="max_length",     # fixed length tak padding
        truncation=True,          # long text ko cut karna
        max_length=128            # maximum sequence length
    )

# poore dataset par tokenization apply karna
tokenized_test = test_dataset.map(
    tokenize_function,
    batched=True
)

# dataset ko PyTorch tensor format mein convert karna
tokenized_test.set_format(
    type="torch",
    columns=["input_ids", "attention_mask"]
)

# confirmation message
print("Tokenization complete!")

Map:   0%|          | 0/63978 [00:00<?, ? examples/s]

Tokenization complete!


In [ ]:
# Trainer class import karna
from transformers import Trainer

# model ke liye trainer object create karna
trainer = Trainer(model=model)

# prediction start hone ka message
print("Running predictions...")

# test dataset par predictions generate karna
predictions_output = trainer.predict(tokenized_test)

# prediction complete hone ka message
print("Predictions complete!")

Running predictions...


Predictions complete!


In [1]:
# purani datasets aur torchvision versions uninstall karna
!pip uninstall -y datasets torchvision
# required versions install karna compatibility ke liye
!pip install datasets==2.21.0 torchvision==0.22.1

Found existing installation: datasets 4.0.0
Uninstalling datasets-4.0.0:
  Successfully uninstalled datasets-4.0.0
Found existing installation: torchvision 0.26.0+cu128
Uninstalling torchvision-0.26.0+cu128:
  Successfully uninstalled torchvision-0.26.0+cu128
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 69.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 821.0/821.0 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.1/393.1 MB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.7/897.7 kB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.0/571.0 MB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.2/200.2 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
# sigmoid function import karna logits ko probabilities mein convert karne ke liye
from scipy.special import expit
# evaluation report generate karne ke liye function import karna
from sklearn.metrics import classification_report

# # logits ko probabilities mein convert karna
probs = expit(predictions_output.predictions)

# threshold 0.5 use karke probabilities ko binary labels mein convert karna
predicted_labels = (probs > 0.5).astype(int)

# class names define karna
target_names = [
    'toxic',
    'severe_toxic',
    'obscene',
    'threat',
    'insult',
    'identity_hate'
]

# heading print karna
print("--- CIVICGUARD MODEL EVALUATION REPORT ---")
# classification report generate aur print karna
print(
    classification_report(
        true_labels,                  # actual labels
        predicted_labels,             # predicted labels
        target_names=target_names,
        zero_division=0               # division by zero errors avoid karna
    )
)

--- CIVICGUARD MODEL EVALUATION REPORT ---
               precision    recall  f1-score   support

        toxic       0.57      0.87      0.69      6090
 severe_toxic       0.30      0.69      0.42       367
      obscene       0.66      0.78      0.71      3691
       threat       0.64      0.50      0.56       211
       insult       0.69      0.71      0.70      3427
identity_hate       0.65      0.63      0.64       712

    micro avg       0.60      0.79      0.68     14498
    macro avg       0.58      0.70      0.62     14498
 weighted avg       0.62      0.79      0.69     14498
  samples avg       0.07      0.07      0.07     14498

